In [68]:
import os

from copy import deepcopy
import pickle

from GANDLF.parseConfig import parseConfig

from GANDLF.data import (
    get_train_loader,
    get_validation_loader,
)

from torch.utils.data import DataLoader

from GANDLF.compute.generic import create_pytorch_objects
from GANDLF.compute.training_loop import train_network
from GANDLF.compute.forward_pass import validate_network
from GANDLF.utils import populate_header_in_parameters, parseTrainingCSV, populate_channel_keys_in_params, send_model_to_device, get_class_imbalance_weights
from GANDLF.data.ImagesFromDataFrame import ImagesFromDataFrame
from GANDLF.models import get_model
from GANDLF.schedulers import get_scheduler
from GANDLF.optimizers import get_optimizer

from GANDLF.utils.write_parse import get_dataframe

In [9]:
# copied from BraTS_internal_loop.py (branch: Brandon_test_GaNDLF at https://github.com/mansishr/openfl.git)

"""
def get_loaders(parameters, train_csv_path=None, val_csv_path=None):
"""
"""
    This function creates the data loaders for each colaborator train, and test data.
    Args:
        parameters (dict): The parameters dictionary.
        train_csv_path (str): The path to the train CSV file.
        test_csv_path (str): The path to the test CSV file.
    Returns:
        train_loader (torch.utils.data.DataLoader): The training data loader.
        test_loader (torch.utils.data.DataLoader): The testing data loader.
"""
"""
    # initialize loaders
    train_loader, val_loader = None, None
    headers_train, headers_val = None, None

    # populate the data frames for the train loader
    if train_csv_path is not None:
        parameters["training_data"], headers_train = parseTrainingCSV(
            train_csv_path, train=True
        )
        parameters = populate_header_in_parameters(parameters, headers_train)

    # get the train loader
        train_loader = get_train_loader(parameters)
        parameters["training_samples_size"] = len(train_loader)

        # Calculate the weights here
        (
            parameters["weights"],
            parameters["class_weights"],
        ) = get_class_imbalance_weights(parameters["training_data"], parameters)
    else:
        raise Exception("Train csv data is required")

    # populate the data frames for the test loader
    if val_csv_path is not None:
        parameters["validation_data"], headers_val = parseTrainingCSV(
            val_csv_path, train=False
        )

        if headers_train is None:
            parameters = populate_header_in_parameters(
                    parameters, headers_val
                )

        # get the validation loader
        val_loader = get_validation_loader(parameters)
    else:
        raise Exception("Validation csv data is required")

    return train_loader, val_loader, parameters
    
"""


'\n    # initialize loaders\n    train_loader, val_loader = None, None\n    headers_train, headers_val = None, None\n\n    # populate the data frames for the train loader\n    if train_csv_path is not None:\n        parameters["training_data"], headers_train = parseTrainingCSV(\n            train_csv_path, train=True\n        )\n        parameters = populate_header_in_parameters(parameters, headers_train)\n\n    # get the train loader\n        train_loader = get_train_loader(parameters)\n        parameters["training_samples_size"] = len(train_loader)\n\n        # Calculate the weights here\n        (\n            parameters["weights"],\n            parameters["class_weights"],\n        ) = get_class_imbalance_weights(parameters["training_data"], parameters)\n    else:\n        raise Exception("Train csv data is required")\n\n    # populate the data frames for the test loader\n    if val_csv_path is not None:\n        parameters["validation_data"], headers_val = parseTrainingCSV(\n     

In [3]:
data_pardir = '/raid/datasets/GaNDLF_FeTS2022_Collab_TrainValData'

train_data_path = os.path.join(data_pardir, 'seg_test_train_small.csv')
valid_data_path = os.path.join(data_pardir, 'seg_test_val_small.csv')

gandlf_config_path = '/home/edwardsb/repositories/MansiOpenFL/openfl-tutorials/experimental/GaNDLF_tutorials/GaNDLF-config/config_segmentation_BraTS.yaml'

In [4]:
"""
Ok I just did a test where I changed the train parameter to False in the line, 

return DataLoader(
        ImagesFromDataFrame(
            get_dataframe(params["training_data"]),
            params,
            train=False,
            loader_type="train",
        ),
        batch_size=params["batch_size"],
        shuffle=True,
        pin_memory=False,  # params["pin_memory_dataloader"], # this is going OOM if True - needs investigation
    )
 of get_train_loader in GaNDLF.GANDLF.__init__.py
"""

'\nOk I just did a test where I changed the train parameter to False in the line, \n\nreturn DataLoader(\n        ImagesFromDataFrame(\n            get_dataframe(params["training_data"]),\n            params,\n            train=False,\n            loader_type="train",\n        ),\n        batch_size=params["batch_size"],\n        shuffle=True,\n        pin_memory=False,  # params["pin_memory_dataloader"], # this is going OOM if True - needs investigation\n    )\n of get_train_loader in GaNDLF.GANDLF.__init__.py\n'

In [ ]:
gandlf_config = parseConfig(gandlf_config_path)

In [15]:
params=gandlf_config

patches_queue_TFalse = ImagesFromDataFrame(
            get_dataframe(params["training_data"]),
            params,
            train=False,
            loader_type="train",
        )

Constructing queue for train data: 100%|██████████| 2/2 [00:00<00:00,  3.43it/s]

Brandon DEBUG: transform is: Compose([ZNormalization(masking_method=None)])


In [16]:
patches_queue_TTrue = ImagesFromDataFrame(
            get_dataframe(params["training_data"]),
            params,
            train=True,
            loader_type="train",
        )

Constructing queue for train data: 100%|██████████| 2/2 [00:00<00:00,  3.41it/s]

Brandon DEBUG: transform is: Compose([ZNormalization(masking_method=None)])


In [18]:
pickle.dumps(patches_queue_TTrue)

NotImplementedError: ('{} cannot be pickled', '_SingleProcessDataLoaderIter')

In [19]:
pickle.dumps(patches_queue_TFalse)

In [28]:
possible_attributes = set(patches_queue_TTrue.__dir__()).difference(set(patches_queue_TFalse.__dir__()))

In [37]:
for att in possible_attributes:
    sesser = ImagesFromDataFrame(
            get_dataframe(params["training_data"]),
            params,
            train=True,
            loader_type="train",
        )
    setattr(sesser, att, None)
    print(f"\nTrying to pickle without attribute {att}")
    print('\n\n')
    try:
        pickle.dumps(sesser)
        print("##############################################")
        print(f"Succesfully pikled without attribute: {att}")
        print("##############################################")
    except NotImplementedError as e:
        print(f"Cannot pickle by removing attribute: {att}")
        

Constructing queue for train data: 100%|██████████| 2/2 [00:00<00:00,  3.24it/s]


Brandon DEBUG: transform is: Compose([ZNormalization(masking_method=None)])

Trying to pickle without attribute get_max_memory_pretty



Cannot pickle by removing attribute: get_max_memory_pretty


Constructing queue for train data: 100%|██████████| 2/2 [00:00<00:00,  3.37it/s]


Brandon DEBUG: transform is: Compose([ZNormalization(masking_method=None)])

Trying to pickle without attribute _print



Cannot pickle by removing attribute: _print


Constructing queue for train data: 100%|██████████| 2/2 [00:00<00:00,  3.53it/s]


...

Constructing queue for train data: 100%|██████████| 2/2 [00:00<00:00,  5.25it/s]


Brandon DEBUG: transform is: Compose([ZNormalization(masking_method=None)])

Trying to pickle without attribute patches_list



Cannot pickle by removing attribute: patches_list


Constructing queue for train data: 100%|██████████| 2/2 [00:00<00:00,  5.15it/s]

Brandon DEBUG: transform is: Compose([ZNormalization(masking_method=None)])


AttributeError: can't set attribute

In [82]:
sesser = ImagesFromDataFrame(
            get_dataframe(params["training_data"]),
            params,
            train=True,
            loader_type="train",
        )

Constructing queue for train data: 100%|██████████| 2/2 [00:00<00:00,  2.92it/s]

Brandon DEBUG: transform is: Compose([ZNormalization(masking_method=None)])


In [87]:
type(sesser._subjects_iterable)

torch.utils.data.dataloader._SingleProcessDataLoaderIter

In [83]:
for subject in sesser._subjects_iterable:
    example_subject = subject
    deepcopy(subject)

In [84]:
subject.keys()

dict_keys(['subject_id', '1', 'spacing', '2', '3', '4', 'label', 'path_to_metadata'])

In [85]:
for key, value in subject.items():
    print(key)
    print(value)
    deepcopy(value)
    print()
    
    

subject_id
FeTS2022_01342

1
ScalarImage(shape: (1, 240, 240, 155); spacing: (1.00, 1.00, 1.00); orientation: LPS+; dtype: torch.FloatTensor; memory: 34.1 MiB)

spacing
tensor([1., 1., 1.])

2
ScalarImage(shape: (1, 240, 240, 155); spacing: (1.00, 1.00, 1.00); orientation: LPS+; dtype: torch.FloatTensor; memory: 34.1 MiB)

3
ScalarImage(shape: (1, 240, 240, 155); spacing: (1.00, 1.00, 1.00); orientation: LPS+; dtype: torch.FloatTensor; memory: 34.1 MiB)

4
ScalarImage(shape: (1, 240, 240, 155); spacing: (1.00, 1.00, 1.00); orientation: LPS+; dtype: torch.FloatTensor; memory: 34.1 MiB)

label
LabelMap(shape: (1, 240, 240, 155); spacing: (1.00, 1.00, 1.00); orientation: LPS+; dtype: torch.IntTensor; memory: 34.1 MiB)

path_to_metadata
/raid/datasets/FeTS22/MICCAI_FeTS2022_TrainingData/FeTS2022_01342/FeTS2022_01342_seg.nii.gz



In [86]:
list(sesser._subjects_iterable)

[]

In [81]:
deepcopy(list(sesser._subjects_iterable))

[]

In [73]:
deepcopy(sesser._subjects_iterable)

NotImplementedError: ('{} cannot be pickled', '_SingleProcessDataLoaderIter')

In [65]:
sesser

Queue(max_length=100, num_subjects=2, num_patches=0, samples_per_volume=40, num_sampled_patches=0, iterations_per_epoch=80)

In [69]:
new_train_loader = DataLoader(
        sesser,
        batch_size=params["batch_size"],
        shuffle=True,
        pin_memory=False,  # params["pin_memory_dataloader"], # this is going OOM if True - needs investigation
    )

In [70]:
for subject in new_train_loader:
    print(subject)

/home/edwardsb/virtual/MansiOpenFL/lib/python3.8/site-packages/torchio/data/queue.py:215: RuntimeWarning: Queue length (100) not divisible by the number of patches per volume (40)
  warnings.warn(message, RuntimeWarning)


{'subject_id': ['FeTS2022_01342'], '1': {'data': tensor([[[[[-0.4068, -0.4068, -0.4068,  ..., -0.4068, -0.4068, -0.4068],
           [-0.4068, -0.4068, -0.4068,  ..., -0.4068, -0.4068, -0.4068],
           [-0.4068, -0.4068, -0.4068,  ..., -0.4068, -0.4068, -0.4068],
           ...,
           [ 2.5194,  2.5109,  2.4430,  ...,  2.7738,  2.9265,  3.2403],
           [ 2.2819,  2.3921,  2.3328,  ...,  2.6466,  2.8926,  3.1894],
           [ 1.9341,  2.2649,  2.2649,  ...,  2.5278,  2.9943,  3.1216]],

          [[-0.4068, -0.4068, -0.4068,  ..., -0.4068, -0.4068, -0.4068],
           [-0.4068, -0.4068, -0.4068,  ..., -0.4068, -0.4068, -0.4068],
           [-0.4068, -0.4068, -0.4068,  ..., -0.4068, -0.4068, -0.4068],
           ...,
           [ 2.3837,  2.4346,  2.3752,  ...,  1.7136,  2.3243,  2.7738],
           [ 2.3582,  2.3667,  2.1971,  ...,  1.9426,  2.5703,  2.7908],
           [ 2.3837,  2.3497,  2.0189,  ...,  2.2055,  2.7653,  2.8417]],

          [[-0.4068, -0.4068, -0.4068, 

{'subject_id': ['FeTS2022_01342'], '1': {'data': tensor([[[[[-0.4068, -0.4068, -0.4068,  ..., -0.4068, -0.4068, -0.4068],
           [-0.4068, -0.4068, -0.4068,  ..., -0.4068, -0.4068, -0.4068],
           [-0.4068, -0.4068, -0.4068,  ..., -0.4068, -0.4068, -0.4068],
           ...,
           [ 3.0368,  3.2827,  3.2658,  ..., -0.4068, -0.4068, -0.4068],
           [ 3.0876,  3.0452,  3.2742,  ..., -0.4068, -0.4068, -0.4068],
           [ 2.7908,  2.8417,  3.0452,  ..., -0.4068, -0.4068, -0.4068]],

          [[-0.4068, -0.4068, -0.4068,  ..., -0.4068, -0.4068, -0.4068],
           [-0.4068, -0.4068, -0.4068,  ..., -0.4068, -0.4068, -0.4068],
           [-0.4068, -0.4068, -0.4068,  ..., -0.4068, -0.4068, -0.4068],
           ...,
           [ 3.1385,  3.4184,  3.2573,  ..., -0.4068, -0.4068, -0.4068],
           [ 3.1046,  3.1301,  3.2827,  ..., -0.4068, -0.4068, -0.4068],
           [ 3.3167,  3.2488,  3.2742,  ..., -0.4068, -0.4068, -0.4068]],

          [[-0.4068, -0.4068, -0.4068, 

{'subject_id': ['FeTS2022_01332'], '1': {'data': tensor([[[[[ 2.3987,  2.2349,  2.1905,  ...,  2.9163,  2.4351,  2.0752],
           [ 2.2653,  2.0853,  2.0206,  ...,  2.6049,  2.2572,  1.9600],
           [ 2.2915,  2.1622,  2.0469,  ...,  2.5968,  2.1338,  1.3696],
           ...,
           [ 3.5774,  3.3914,  3.1973,  ..., -0.4297, -0.4297, -0.4297],
           [ 3.4278,  3.1205,  2.9203,  ..., -0.4297, -0.4297, -0.4297],
           [ 2.7485,  2.4775,  2.0166,  ..., -0.4297, -0.4297, -0.4297]],

          [[ 2.3926,  2.1581,  1.9074,  ...,  2.9789,  2.5807,  2.3300],
           [ 2.2754,  2.2996,  2.2693,  ...,  2.6453,  2.3178,  2.1702],
           [ 2.3077,  2.2390,  2.3482,  ...,  2.6110,  2.3663,  2.0206],
           ...,
           [ 3.4096,  3.2943,  3.2923,  ..., -0.4297, -0.4297, -0.4297],
           [ 3.4702,  3.2600,  3.1730,  ..., -0.4297, -0.4297, -0.4297],
           [ 3.2620,  2.9688,  2.8637,  ..., -0.4297, -0.4297, -0.4297]],

          [[ 2.3502,  1.9580,  1.5940, 

{'subject_id': ['FeTS2022_01332'], '1': {'data': tensor([[[[[ 1.3696,  1.5940,  1.6082,  ..., -0.4297, -0.4297, -0.4297],
           [ 1.6365,  1.9478,  1.8933,  ..., -0.4297, -0.4297, -0.4297],
           [ 2.0024,  2.0793,  2.0449,  ..., -0.4297, -0.4297, -0.4297],
           ...,
           [ 2.3482,  2.2329,  2.4310,  ..., -0.4297, -0.4297, -0.4297],
           [ 1.7760,  2.1318,  2.3603,  ..., -0.4297, -0.4297, -0.4297],
           [ 1.7821,  1.9175,  2.1197,  ..., -0.4297, -0.4297, -0.4297]],

          [[ 1.2625,  1.3898,  1.2706,  ..., -0.4297, -0.4297, -0.4297],
           [ 1.4242,  1.2241,  1.0927,  ..., -0.4297, -0.4297, -0.4297],
           [ 1.5091,  1.0583,  1.2685,  ..., -0.4297, -0.4297, -0.4297],
           ...,
           [ 2.3926,  2.4230,  2.6373,  ..., -0.4297, -0.4297, -0.4297],
           [ 1.9964,  2.1702,  2.4513,  ..., -0.4297, -0.4297, -0.4297],
           [ 1.7255,  1.6507,  1.9317,  ..., -0.4297, -0.4297, -0.4297]],

          [[ 0.9693,  0.5973,  0.3244, 

{'subject_id': ['FeTS2022_01332'], '1': {'data': tensor([[[[[ 0.4194,  0.1808,  0.1040,  ..., -0.4297, -0.4297, -0.4297],
           [ 0.1667,  0.1950,  0.1970,  ..., -0.4297, -0.4297, -0.4297],
           [ 0.1060,  0.0191,  0.0029,  ..., -0.4297, -0.4297, -0.4297],
           ...,
           [ 2.1541,  2.2936,  2.7586,  ..., -0.4297, -0.4297, -0.4297],
           [ 2.1197,  2.2632,  2.6251,  ..., -0.4297, -0.4297, -0.4297],
           [ 2.1702,  2.2410,  2.5079,  ..., -0.4297, -0.4297, -0.4297]],

          [[ 0.2536,  0.2799,  0.2819,  ..., -0.4297, -0.4297, -0.4297],
           [ 0.1263,  0.2011,  0.2395,  ..., -0.4297, -0.4297, -0.4297],
           [ 0.1950,  0.0616,  0.0191,  ..., -0.4297, -0.4297, -0.4297],
           ...,
           [ 2.0146,  2.1035,  2.5726,  ..., -0.4297, -0.4297, -0.4297],
           [ 2.0328,  2.0894,  2.3178,  ..., -0.4297, -0.4297, -0.4297],
           [ 2.1136,  2.0590,  2.1844,  ..., -0.4297, -0.4297, -0.4297]],

          [[ 0.2981,  0.3305,  0.5043, 

In [60]:
pulled_attributes = {}

for att in possible_attributes:
    if att in ['_fill', 'iterations_per_epoch', 
               'subjects_iterable', 
               '_initialize_subjects_iterable', 
              'get_max_memory', 
              'num_subjects', 
              'num_patches']:
        print(f"######### Skipping the {att} attribute ############")      
    else:
        print(f'Attribute is: {att}')
        print(f"{att} was previously: {getattr(sesser, att)}")
        pulled_attributes[att] = getattr(sesser, att)
        setattr(sesser, att, None)
    
pickle.dumps(sesser)

Constructing queue for train data: 100%|██████████| 2/2 [00:00<00:00,  3.44it/s]

Brandon DEBUG: transform is: Compose([ZNormalization(masking_method=None)])
Attribute is: get_max_memory_pretty
get_max_memory_pretty was previously: <bound method Queue.get_max_memory_pretty of Queue(max_length=100, num_subjects=2, num_patches=0, samples_per_volume=40, num_sampled_patches=0, iterations_per_epoch=80)>
Attribute is: _print
_print was previously: <bound method Queue._print of Queue(max_length=100, num_subjects=2, num_patches=0, samples_per_volume=40, num_sampled_patches=0, iterations_per_epoch=80)>
Attribute is: _get_subjects_iterable
_get_subjects_iterable was previously: <bound method Queue._get_subjects_iterable of Queue(max_length=100, num_subjects=2, num_patches=0, samples_per_volume=40, num_sampled_patches=0, iterations_per_epoch=80)>
Attribute is: shuffle_patches
shuffle_patches was previously: True
Attribute is: _get_next_subject
_get_next_subject was previously: <bound method Queue._get_next_subject of Queue(max_length=100, num_subjects=2, num_patches=0, samples

b'\x80\x04\x95N\x01\x00\x00\x00\x00\x00\x00\x8c\x12torchio.data.queue\x94\x8c\x05Queue\x94\x93\x94)\x81\x94}\x94(\x8c\x10subjects_dataset\x94N\x8c\nmax_length\x94N\x8c\x10shuffle_subjects\x94N\x8c\x0fshuffle_patches\x94N\x8c\x12samples_per_volume\x94N\x8c\x07sampler\x94N\x8c\x0bnum_workers\x94N\x8c\x07verbose\x94N\x8c\x12_subjects_iterable\x94N\x8c\x0cpatches_list\x94N\x8c\x13num_sampled_patches\x94N\x8c\x15get_max_memory_pretty\x94N\x8c\x06_print\x94N\x8c\x16_get_subjects_iterable\x94N\x8c\x11_get_next_subject\x94N\x8c\x0f_get_first_item\x94Nub.'

In [58]:
for att, value in pulled_attributes.items():
    if att not in ["_subjects_iterable"]:
        print(f"Putting back in {att}")
        temp = deepcopy(sesser)
        setattr(temp, att, value)
        pickle.dumps(temp)


Putting back in get_max_memory_pretty
Putting back in _print
Putting back in _get_subjects_iterable
Putting back in shuffle_patches
Putting back in _get_next_subject
Putting back in samples_per_volume
Putting back in shuffle_subjects
Putting back in subjects_dataset
Putting back in max_length
Putting back in sampler
Putting back in patches_list
Putting back in _get_first_item
Putting back in num_sampled_patches
Putting back in num_workers
Putting back in verbose


In [26]:
"Here are the attributes for which pickling could not happen after put back in:
_subjects_iterable

"

In [63]:
sesser

Queue(max_length=100, num_subjects=2, num_patches=0, samples_per_volume=40, num_sampled_patches=0, iterations_per_epoch=80)

In [5]:
train_loader, val_loader, parameters = get_loaders(parameters=gandlf_config, 
                                                   train_csv_path=train_data_path, 
                                                   val_csv_path=valid_data_path)

Constructing queue for train data: 100%|██████████| 2/2 [00:00<00:00,  2.97it/s]


Brandon DEBUG: transform is: Compose([ZNormalization(masking_method=None)])
Calculating weights


Constructing queue for penalty data: 100%|██████████| 2/2 [00:00<00:00,  3.90it/s]


Brandon DEBUG: transform is: Compose([ZNormalization(masking_method=None)])


Looping over training data for penalty calculation: 100%|██████████| 2/2 [00:01<00:00,  1.36it/s]
Constructing queue for validation data: 100%|██████████| 1/1 [00:00<00:00,  4.71it/s]


Brandon DEBUG: transform is: Compose([ZNormalization(masking_method=None)])


In [5]:
deepcopy(train_loader)

In [ ]:
# ok, so changing the train parameter to false in ImagesFromDataframe allows it to be pickled (you may
# not see this in the output above as it requires the supporting files to be changed)


# So it is in the patches_queue (this is what is returned by ImagesToDataframe)






In [ ]:
deepcopy(val_loader)

In [35]:
pickle.dumps(val_loader)

In [33]:
pickle.dumps(train_loader)

NotImplementedError: ('{} cannot be pickled', '_SingleProcessDataLoaderIter')

In [ ]:
# Below just to see if the train and val loader have different attribute lists, but they do not

In [6]:
train_loader.__dir__()

['dataset',
 'num_workers',
 'prefetch_factor',
 'pin_memory',
 'timeout',
 'worker_init_fn',
 '_DataLoader__multiprocessing_context',
 '_dataset_kind',
 'batch_size',
 'drop_last',
 'sampler',
 'batch_sampler',
 'generator',
 'collate_fn',
 'persistent_workers',
 '_DataLoader__initialized',
 '_IterableDataset_len_called',
 '_iterator',
 '__module__',
 '__annotations__',
 '__doc__',
 '__init__',
 '_get_iterator',
 'multiprocessing_context',
 '__setattr__',
 '__iter__',
 '_auto_collation',
 '_index_sampler',
 '__len__',
 'check_worker_number_rationality',
 '__orig_bases__',
 '__dict__',
 '__weakref__',
 '__parameters__',
 '__slotnames__',
 '__slots__',
 '_is_protocol',
 '__new__',
 '__class_getitem__',
 '__init_subclass__',
 '__repr__',
 '__hash__',
 '__str__',
 '__getattribute__',
 '__delattr__',
 '__lt__',
 '__le__',
 '__eq__',
 '__ne__',
 '__gt__',
 '__ge__',
 '__reduce_ex__',
 '__reduce__',
 '__subclasshook__',
 '__format__',
 '__sizeof__',
 '__dir__',
 '__class__']

In [8]:
set(train_loader.__dir__()).difference(set(val_loader.__dir__()))

set()

In [ ]:
"""
Bringing in the get_train_loader function to manually play a bit there
"""

from torch.utils.data import DataLoader

from .ImagesFromDataFrame import ImagesFromDataFrame
from GANDLF.utils.write_parse import get_dataframe
from GANDLF.utils import populate_channel_keys_in_params


def get_train_loader(params):
    """
    Get the training data loader.

    Args:
        params (dict): Dictionary of parameters.

    Returns:
        torch.utils.data.DataLoader: The training loader.
    """
    print(f"\n\nBrandon DEBUG -- temporarily changed train to false in train loader\n\n")
    return DataLoader(
        ImagesFromDataFrame(
            get_dataframe(params["training_data"]),
            params,
            train=False,
            loader_type="train",
        ),
        batch_size=params["batch_size"],
        shuffle=True,
        pin_memory=False,  # params["pin_memory_dataloader"], # this is going OOM if True - needs investigation
    )